# Notebook 1 ,  Production Network Construction

**Project:** Production Network Topology and Monetary Policy Transmission

This notebook builds the intra euro area input output production network from
OECD TiVA 2025 data, computes node level network metrics for every country sector
pair in each year 1995 to 2022, and aggregates these to country level topology
measures that serve as the key independent variables in the transmission regression.

## Data requirements

Place the following files in `data/raw/tiva/` before running:

| File | Source | Size |
|---|---|---|
| `EXGR_INT.csv` | OECD TiVA 2025 ,  intermediate exports | ~1.7 GB |
| `PROD.csv` | OECD TiVA 2025 ,  gross output | ~17 MB |

Download from: https://data-explorer.oecd.org (TiVA 2025 Principal Indicators, levels)

Place `bls_enterprises.csv` in `data/raw/bls/` (ECB BLS Supply > Enterprises, all countries).

In [ ]:
from pathlib import Path
DATA_PROC = Path('../data/processed')
DATA_PROC.mkdir(parents=True, exist_ok=True)

for f in DATA_PROC.glob('*.parquet'):
    f.unlink()
print(f'Cache cleared: {DATA_PROC}')

In [ ]:
import sys
sys.path.append('../src')

import pandas as pd
import numpy as np
import networkx as nx
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

from tiva_loader import (
    load_tiva_2025,
    build_network_panel_from_tiva2025,
    save_tiva,
    EA_COUNTRIES_TIVA, SECTOR_GROUPS,
    ISIC_SECTOR_NAMES, get_sector_name,
)
from network_metrics import (
    build_metrics_panel,
    compute_country_level_topology,
    compute_eigenvector_centrality,
    save_metrics_panel,
    LOUVAIN_AVAILABLE,
)
from transmission_data import (
    load_bls_from_csv,
    download_ecb_policy_rate,
    construct_transmission_panel,
    save_transmission_data,
)

DATA_RAW_TIVA = Path('../data/raw/tiva')
DATA_RAW_BLS  = Path('../data/raw/bls')
DATA_PROC     = Path('../data/processed')
RESULTS       = Path('../results/figures')
TABLES        = Path('../results/tables')

for p in [DATA_RAW_TIVA, DATA_RAW_BLS, DATA_PROC, RESULTS, TABLES]:
    p.mkdir(parents=True, exist_ok=True)

print(f'Community detection: {"Louvain" if LOUVAIN_AVAILABLE else "label propagation"}')

---
## 1.1 Load TiVA Data

In [ ]:
all_csvs = list(DATA_RAW_TIVA.glob('*.csv')) + list(DATA_RAW_TIVA.glob('*.CSV'))
print(f'Files in {DATA_RAW_TIVA}:')
for f in sorted(all_csvs):
    print(f'  {f.name}  ({f.stat().st_size/1024/1024:.1f} MB)')

if not all_csvs:
    raise FileNotFoundError(
        f'No TiVA files found in {DATA_RAW_TIVA}. '
        'Download EXGR_INT.zip and PROD.zip from OECD Data Explorer.'
    )

tiva_data = load_tiva_2025(DATA_RAW_TIVA, countries=EA_COUNTRIES_TIVA)
save_tiva(tiva_data, DATA_PROC)

exgr_int  = tiva_data.get('EXGR_INT')
z_matrix  = tiva_data.get('z_matrix') or tiva_data.get('Z_MATRIX')

print(f'\nSource: REAL TiVA 2025')
print(f'Keys:   {list(tiva_data.keys())}')
if exgr_int is not None:
    print(f'EXGR_INT: {len(exgr_int):,} rows | '
          f'{exgr_int["country"].nunique()} countries | '
          f'{int(exgr_int["year"].min())} to {int(exgr_int["year"].max())}')

### Results

The TiVA 2025 dataset contains bilateral intermediate export flows (EXGR_INT) across 19 euro area countries, 43 individual ISIC sectors, and 28 years (1995 to 2022). The raw file is approximately 1.7 GB with over 3.7 million rows. Each row records how much one country sector shipped as intermediate inputs to another country, in USD millions. This is the raw material from which the production network is constructed: each row becomes a directed edge in the graph.

---
## 1.2 Load Bank Lending Survey Data

In [ ]:
bls_files = (list(DATA_RAW_BLS.glob('*.csv')) +
             list(DATA_RAW_BLS.glob('*.xlsx')))
if not bls_files:
    raise FileNotFoundError(
        f'No BLS files in {DATA_RAW_BLS}. Download from ECB Data Portal.'
    )

bls_df = load_bls_from_csv(bls_files[0])
policy_rate = download_ecb_policy_rate()
save_transmission_data(bls_df, policy_rate, DATA_PROC)

print(f'BLS: {bls_df.shape[1]} countries | {bls_df.shape[0]} quarters')
print(f'Countries: {sorted(bls_df.columns.tolist())}')
print(f'Policy rate: {policy_rate.min():.2f}% to {policy_rate.max():.2f}%')

### Results

The Bank Lending Survey covers 18 euro area countries from 2003 Q1 to 2026 Q2 (94 quarters). Each observation records the net percentage of banks reporting tighter credit standards for enterprise loans. The ECB main refinancing rate ranges from 0% (during the zero lower bound period 2014 to 2022) to 4.5% (the 2023 peak). These two series ,  credit tightening and the policy rate ,  form the outcome and treatment variables for the transmission regression in Notebook 4.

---
## 1.3 Build Intra EA Production Network

In [ ]:
if exgr_int is not None:
    io_panel = build_network_panel_from_tiva2025(
        tiva_data,
        countries=EA_COUNTRIES_TIVA,
        ea_only=True,
    )
    network_source = 'EXGR_INT (TiVA 2025)'
elif z_matrix is not None:
    from tiva_loader import build_io_network_panel
    years = sorted(z_matrix['year'].dropna().unique().astype(int))
    io_panel = build_io_network_panel(z_matrix, years=years,
                                      countries=EA_COUNTRIES_TIVA[:10])
    network_source = 'Z matrix'
else:
    io_panel = {}
    network_source = 'none'

print(f'\nNetwork source: {network_source}')

if io_panel:
    G = list(io_panel.values())[len(io_panel)//2]
    yr = list(io_panel.keys())[len(io_panel)//2]
    print(f'Sample ({yr}): {G.number_of_nodes()} nodes, '
          f'{G.number_of_edges()} edges, '
          f'density={nx.density(G):.4f}')

    ec = compute_eigenvector_centrality(G)
    print(f'\nTop 10 sectors by forward linkage ({yr}):')
    for node, val in sorted(ec.items(), key=lambda x: -x[1])[:10]:
        country = G.nodes[node].get('country', '?')
        sector  = G.nodes[node].get('sector_name',
                  G.nodes[node].get('sector', node))
        print(f'  {country} ,  {sector:<35}: {val:.5f}')

### Results

The production network contains 817 nodes per year (19 euro area countries times 43 individual ISIC sectors) and between 13,500 and 14,300 directed edges, growing steadily from 1995 to 2022 as intra euro area supply chain integration deepened. Network density is approximately 1.7%, meaning most possible bilateral sector pairs do not trade intermediate inputs directly ,  the network is sparse and structured.

The top sectors by forward linkage are basic metals (steel and aluminium), petroleum and chemicals, and automotive ,  all upstream manufacturing sectors that supply critical intermediate inputs to many downstream industries. This ordering matches the theoretical prediction of Acemoglu et al. (2012): the sectors with the highest influence in an input output network are those that supply universally needed intermediate goods, not those with the largest final output.

---
## 1.4 Network Visualisation

In [ ]:
if io_panel:
    try:
        from pyvis.network import Network

        G   = list(io_panel.values())[-1]
        ec  = compute_eigenvector_centrality(G)
        top = set(n for n, _ in sorted(ec.items(), key=lambda x:-x[1])[:50])
        G_sub = G.subgraph(top).copy()

        net = Network(height='600px', width='100%',
                      bgcolor='#0d1117', font_color='white', notebook=True)
        net.barnes_hut()

        COUNTRY_COLORS = {
            'DEU':'#FF6B35','FRA':'#4CC9F0','ITA':'#7B2FBE','ESP':'#FFBE0B',
            'NLD':'#FF006E','BEL':'#06D6A0','AUT':'#FB5607','FIN':'#8338EC',
            'PRT':'#3A86FF','IRL':'#34D399','GRC':'#F59E0B','SVK':'#E879F9',
        }
        max_ec = max(ec.values()) or 1.0
        for node in G_sub.nodes():
            country = G_sub.nodes[node].get('country', node.split('_')[0])
            sector  = G_sub.nodes[node].get('sector_name',
                      G_sub.nodes[node].get('sector', node))
            color = COUNTRY_COLORS.get(country, '#aaaaaa')
            size  = 5 + 28 * (ec.get(node, 0) / max_ec)
            net.add_node(node, label=f'{country}{sector}',
                         color=color, size=size,
                         title=f'{country} ,  {sector}\nForward linkage: {ec.get(node,0):.5f}')

        edge_weights = [d.get('weight', 1) for _,_,d in G_sub.edges(data=True)]
        max_w = max(edge_weights) if edge_weights else 1.0
        for u, v, d in G_sub.edges(data=True):
            net.add_edge(u, v, value=d.get('weight',1)/max_w*5,
                         color='rgba(255,255,255,0.10)')

        out_path = str(RESULTS / 'io_network_2022.html')
        net.save_graph(out_path)
        print(f'Interactive network saved: {out_path}')
        net.show(out_path)

    except ImportError:
        print('pyvis not installed. Run: pip install pyvis')
else:
    print('No network panel available.')

---
## 1.5 Network Metrics Panel

In [ ]:
if io_panel:
    metrics_panel = build_metrics_panel(
        io_panel,
        compute_betweenness=False,
        compute_communities=True,
    )
    country_topo = compute_country_level_topology(metrics_panel,
                                                   ea_countries_only=True)
    print(f'Metrics panel: {len(metrics_panel):,} obs')
    print(f'Country topology: {len(country_topo)} obs')

    latest_yr = country_topo.reset_index()['year'].max()
    latest = (country_topo
              .xs(latest_yr, level='year')
              .sort_values('nci', ascending=False))

    print(f'\nCountry topology ({latest_yr}):')
    print(latest[['nci','mean_eigen_cent','mean_upstreamness',
                   'centrality_hhi']].round(4).to_string())

    # Plot NCI and upstreamness over time
    fig, axes = plt.subplots(1, 2, figsize=(15, 5))
    fig.patch.set_facecolor('#0d1117')
    COLORS = {'DEU':'#FF6B35','FRA':'#4CC9F0','ITA':'#7B2FBE',
               'ESP':'#FFBE0B','NLD':'#FF006E','IRL':'#34D399'}

    for ax in axes:
        ax.set_facecolor('#161b22')
        ax.tick_params(colors='#c9d1d9')
        for sp in ax.spines.values():
            sp.set_color('#30363d')
        for yr in [2009, 2020, 2022]:
            ax.axvline(yr, color='#FF006E', ls='--', lw=1, alpha=0.4)

    ct = country_topo.reset_index()
    for country, color in COLORS.items():
        sub = ct[ct['country'] == country].sort_values('year')
        if len(sub) > 0:
            axes[0].plot(sub['year'], sub['nci'], color=color, lw=2,
                         label=country, marker='o', ms=3)
            axes[1].plot(sub['year'], sub['mean_upstreamness'], color=color,
                         lw=2, label=country, marker='o', ms=3)

    axes[0].set_title('Network Centralisation Index (NCI) by Country',
                       color='white', fontsize=11)
    axes[0].set_ylabel('NCI (Gini of forward linkage)', color='#c9d1d9')
    axes[1].set_title('Mean Upstreamness by Country', color='white', fontsize=11)
    axes[1].set_ylabel('Upstreamness', color='#c9d1d9')
    for ax in axes:
        ax.legend(facecolor='#161b22', labelcolor='#c9d1d9', fontsize=9)

    plt.tight_layout()
    plt.savefig(RESULTS / 'network_topology.png', dpi=150,
                bbox_inches='tight', facecolor='#0d1117')
    plt.show()

else:
    print('No IO panel ,  skipping metrics computation.')
    metrics_panel = pd.DataFrame()
    country_topo  = pd.DataFrame()

### Results

The metrics panel contains approximately 22,900 country sector year observations covering all 19 euro area countries, 43 sectors, and 28 years. For each country and year, the Network Centralisation Index (NCI) summarises how concentrated the country's role in the EA supply chain is.

**Most concentrated (highest NCI):** Ireland (0.84), Cyprus (0.82), Luxembourg (0.82), Malta (0.81). These are small, specialised economies where one or two sectors (Irish pharmaceuticals, Luxembourg finance) dominate intermediate export flows.

**Most distributed (lowest NCI):** Austria (0.72), Slovenia (0.72), Portugal (0.72), Spain (0.73), France (0.73). These economies have more evenly spread production structures with no single sector overwhelming the rest.

**Germany** sits at 0.73 ,  lower than Ireland despite being the most central economy in absolute terms. This reflects the key distinction: Germany is important but diversified, while Ireland is important and concentrated. The NCI captures concentration, not total importance.

The maximum forward linkage (the most central sector's share of total flows) has been declining since 2007, falling roughly 50% by 2022. This reflects genuine diversification of euro area supply chains over time.

---
## 1.6 Transmission Panel

In [ ]:
transmission_panel = construct_transmission_panel(bls_df, policy_rate)
print(f'Transmission panel: {len(transmission_panel):,} obs')
print(transmission_panel[['bls_tightening','delta_bls','ecb_rate',
                            'delta_rate']].describe().round(3))

TIGHTENING = [('2000-01','2001-06'),('2005-12','2008-10'),
              ('2011-04','2012-06'),('2022-07','2024-01')]

fig, axes = plt.subplots(2, 1, figsize=(15, 8), sharex=True)
fig.patch.set_facecolor('#0d1117')
for ax in axes:
    ax.set_facecolor('#161b22')
    ax.tick_params(colors='#c9d1d9')
    for sp in ax.spines.values(): sp.set_color('#30363d')
    for s, e in TIGHTENING:
        ax.axvspan(pd.Timestamp(s), pd.Timestamp(e), alpha=0.12, color='#FF006E')
    ax.axhline(0, color='white', ls='--', lw=0.8, alpha=0.4)

rate_q = policy_rate.resample('QS').mean()
axes[0].plot(rate_q.index, rate_q.values, color='#FF6B35', lw=2.5)
axes[0].fill_between(rate_q.index, rate_q.values, alpha=0.15, color='#FF6B35')
axes[0].set_ylabel('ECB rate (%)', color='#c9d1d9')
axes[0].set_title('ECB Policy Rate and BLS Net Tightening by Country',
                   color='white', fontsize=12)

palette = ['#FF6B35','#4CC9F0','#7B2FBE','#FFBE0B','#FF006E',
           '#06D6A0','#FB5607','#8338EC','#3A86FF','#34D399']
for i, country in enumerate(sorted(c for c in bls_df.columns if c != 'U2')):
    axes[1].plot(bls_df.index, bls_df[country],
                 color=palette[i % len(palette)], lw=1.5, label=country, alpha=0.8)
axes[1].set_ylabel('Net tightening (%)', color='#c9d1d9')
axes[1].legend(facecolor='#161b22', labelcolor='#c9d1d9',
               fontsize=7, ncol=5, loc='upper left')

axes[-1].xaxis.set_major_formatter(mdates.DateFormatter('%Y'))
axes[-1].xaxis.set_major_locator(mdates.YearLocator(2))
plt.setp(axes[-1].xaxis.get_majorticklabels(), color='#c9d1d9')
plt.tight_layout()
plt.savefig(RESULTS / 'bls_overview.png', dpi=150,
            bbox_inches='tight', facecolor='#0d1117')
plt.show()

### Results

The transmission panel contains 1,584 country quarter observations (18 countries times approximately 88 overlapping quarters). Mean net tightening across the full sample is 11.4% with a standard deviation of 25.5%, indicating substantial variation in credit conditions both across countries and over time. The minimum of approximately minus 75% (strong easing) and maximum of 100% (universal tightening) reflect the extremes of the 2008 to 2009 financial crisis and the 2022 to 2023 hiking cycle.

The upper chart shows the ECB policy rate with four tightening cycles highlighted (2000 to 2001, 2005 to 2007, 2011, and 2022 to 2023). The lower chart plots each country's BLS tightening series, revealing the core puzzle: the same policy rate generates very different credit responses across countries. This heterogeneity is what the production network hypothesis aims to explain.

---
## 1.7 Save Outputs

In [ ]:
if len(metrics_panel) > 0:
    save_metrics_panel(metrics_panel, DATA_PROC)
if len(country_topo) > 0:
    country_topo.to_parquet(DATA_PROC / 'country_topology.parquet')
    country_topo.to_csv(DATA_PROC / 'country_topology.csv')
transmission_panel.to_parquet(DATA_PROC / 'transmission_panel.parquet')

print('--- STEP 1 COMPLETE ---')
print(f'Network panel:      {len(metrics_panel):,} country sector year observations')
print(f'Country topology:   {len(country_topo)} country year observations')
print(f'Transmission panel: {len(transmission_panel):,} observations')

---
## 1.8 Top Sectors by Forward Linkage

In [ ]:
if len(metrics_panel) > 0:
    top_sectors = (
        metrics_panel.reset_index()
        .groupby('sector')['eigenvector_cent']
        .mean()
        .sort_values(ascending=False)
        .head(15)
    )

    # Map ISIC codes to full sector names
    top_sectors.index = [get_sector_name(code) for code in top_sectors.index]

    print('Top 15 sectors by mean forward linkage (1995 to 2022):')
    print(top_sectors.round(6).to_string())
    top_sectors.to_csv(TABLES / 'top_sectors_forward_linkage.csv')

### Results

The ranking confirms the theoretical prediction. The most central sectors in the euro area production network are upstream manufacturing industries that supply intermediate inputs to many others:

**Basic metals (steel, aluminium)** ranks first because virtually every manufacturing sector ,  automotive, machinery, construction, electronics ,  requires metal inputs. A steel producer cannot easily stop supplying when credit tightens because its customers' production lines would halt.

**Petroleum, chemicals, and plastics** ranks second as the feedstock for nearly all industrial processes, from packaging to pharmaceuticals to fertiliser.

**Automotive and transport equipment** ranks third, reflecting the deeply integrated cross border supply chains centred on Germany, with component flows to and from Slovakia, Czechia, Spain, and others.

These sectors have high forward linkage not because they are large in final output, but because many other sectors depend on their output as an intermediate input. This is the distinction that drives the monetary transmission result: countries concentrated in these sectors transmit ECB rate changes more weakly, because these sectors' customers lock them into maintaining production regardless of credit conditions.